# Batch 19 — Phase 2 validation on Colab

Runs the Phase-2 cached / Thompson / SIMD variants across ITC 2007 sets 1-8.
Same pipeline as `make batch19` locally, but suited to a Colab VM (reboot-safe Drive mount, multi-core parallelism).

**Steps:**
1. Mount Drive (optional, for reboot-safety)
2. Clone repo + build C++ solver with AVX2
3. Run `scripts/run_batch19.sh` with 3 seeds × 5 algos × 8 sets
4. Zip results and download/save to Drive

**Expected runtime:** A100 ~15 min | T4 high-RAM ~30 min | CPU free ~2 hours

In [ ]:
# Optional: mount Drive so results survive a Colab timeout
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/exam_scheduling_batches

In [ ]:
# Clone the repo (replace OWNER/REPO with your fork)
REPO_URL = 'https://github.com/Dialovos/exam-scheduling-algos-analyze'
!rm -rf /content/exam-scheduling
!git clone --depth 1 $REPO_URL /content/exam-scheduling
%cd /content/exam-scheduling

In [ ]:
# Build — requires g++ 11+ (Colab has it), AVX2 (Colab Intel nodes have it)
!sudo apt-get install -y build-essential bc
!make all

In [ ]:
# Batch 19 spec — tweak for budget vs coverage
import os
os.environ['BATCH19_DIR'] = 'results/batch_019_colab'
os.environ['BATCH19_SEEDS'] = '42 43 44'
os.environ['BATCH19_ALGOS'] = 'tabu_cached sa_cached gd_cached lahc_cached alns_thompson'
os.environ['BATCH19_SETS']  = 'exam_comp_set1 exam_comp_set2 exam_comp_set3 exam_comp_set4 exam_comp_set5 exam_comp_set6 exam_comp_set7 exam_comp_set8'

!bash scripts/run_batch19.sh "$BATCH19_DIR" "$BATCH19_SEEDS" "$BATCH19_ALGOS" "$BATCH19_SETS"

In [ ]:
# Summary table
!python3 scripts/summarize_batch19.py results/batch_019_colab

In [ ]:
# Zip results, upload to Drive if mounted
!zip -qr batch_019_colab.zip results/batch_019_colab
import shutil, os
if os.path.exists('/content/drive/MyDrive/exam_scheduling_batches'):
    shutil.copy('batch_019_colab.zip', '/content/drive/MyDrive/exam_scheduling_batches/')
    print('Copied to Drive.')
# Also download locally via Colab's file UI:
from google.colab import files
files.download('batch_019_colab.zip')